# 🎬 ComfyUI + WAN 2.2 Video Gen — Kaggle (T4 x2)

Spins up ComfyUI on a Kaggle GPU and exposes it through a public Cloudflare tunnel, so you get the full web UI in your own browser.

| Step | What to do |
|---|---|
| 1 | Turn on GPU: **Settings → Accelerator → GPU T4 x2** |
| 2 | Run **Cells 1–5** top to bottom (setup → model download → tunnel) |
| 3 | Copy the URL printed by Cell 5 (`https://xxxx.trycloudflare.com`) → open in a **new tab** |
| 4 | In ComfyUI, set your I2V/T2V workflow nodes: |
| | • Diffusion model → `wan2.2_ti2v_5B_fp16.safetensors` |
| | • VAE → `wan2.2_vae.safetensors` |
| | • Text encoder/CLIP → `umt5_xxl_fp8_e4m3fn_scaled.safetensors` |
| 5 | Run **Cell 6 (keep-alive)** last, and leave it running |

⚠️ **Heads up:** Kaggle sessions can disconnect on their own after a while (idle timeout, connection drop, etc.), even with this cell running — it happens.

**If it disconnects:** re-run Cell 5 to get a fresh tunnel URL, open it in a new tab, then re-run this cell.

💡 **Suggestion:** if you want this handled automatically instead of manually re-running cells, you could wrap Cell 5's tunnel logic in a retry loop inside this cell — so it detects a dropped tunnel and restarts it on its own instead of just printing keep-alive messages.

🔗 The tunnel URL is random and temporary — it changes on every rerun of Cell 5.

# Cell 1 — Clone ComfyUI + install

In [1]:
import os, sys, subprocess

print("⚡ [1/5] Instalando dependencias base y ComfyUI en /kaggle/working...")
subprocess.run(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "aria2", "-qq"], stdout=subprocess.DEVNULL)

os.chdir("/kaggle/working")
if not os.path.exists("/kaggle/working/ComfyUI"):
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", "/kaggle/working/ComfyUI"], check=True)

os.chdir("/kaggle/working/ComfyUI")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=True)

os.chdir("/kaggle/working/ComfyUI/custom_nodes")
nodes = [
    ("ComfyUI-Manager", "https://github.com/ltdrdata/ComfyUI-Manager.git"),
    ("ComfyUI-VideoHelperSuite", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git")
]
for name, repo in nodes:
    if not os.path.exists(name):
        subprocess.run(["git", "clone", repo, name, "-q"], check=True)

os.chdir("/kaggle/working/ComfyUI")
print("✅ ComfyUI y nodos instalados correctamente en /kaggle/working/ComfyUI.")


Cloning into 'ComfyUI'...
remote: Enumerating objects: 47721, done.
remote: Counting objects: 100% (562/562), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 47721 (delta 471), reused 400 (delta 400), pack-reused 47159 (from 2)
Receiving objects: 100% (47721/47721), 93.62 MiB | 26.62 MiB/s, done.
Resolving deltas: 100% (32330/32330), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.3/342.3 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 MB 28.8 MB/s eta 0:00:00
 

# Cell 2 — Download models

In [2]:
import os, subprocess

print("📥 [2/5] Descarga paralela multihilo de Wan 2.2 TI2V 5B (16.9 GB en total)...")
os.chdir("/kaggle/working/ComfyUI")
os.makedirs("models/diffusion_models", exist_ok=True)
os.makedirs("models/vae", exist_ok=True)
os.makedirs("models/text_encoders", exist_ok=True)
os.makedirs("input", exist_ok=True)

downloads = [
    (
        "models/diffusion_models",
        "wan2.2_ti2v_5B_fp16.safetensors",
        "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors"
    ),
    (
        "models/vae",
        "wan2.2_vae.safetensors",
        "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors"
    ),
    (
        "models/text_encoders",
        "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
        "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"
    )
]

for out_dir, filename, url in downloads:
    dest = os.path.join(out_dir, filename)
    if not os.path.exists(dest):
        print(f"🚀 Descargando {filename} a máxima velocidad...")
        cmd = ["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--file-allocation=none", "-d", out_dir, "-o", filename, url]
        subprocess.run(cmd, check=True)
    else:
        print(f"✔️ {filename} ya está descargado.")

# Inyectar imagen de prueba de la modelo de ropa
img_data = "..."
import base64
with open("input/clothing_model.png", "wb") as f:
    f.write(base64.b64decode(img_data))
print("✅ Modelos Wan 2.2 e imagen input/clothing_model.png listos!")


models/diffusion_mo 100%[===================>]   9.31G  95.0MB/s    in 97s     
models/vae/wan2.2_v 100%[===================>]   1.31G   102MB/s    in 12s     
models/text_encoder 100%[===================>]   6.27G  99.2MB/s    in 65s     
models/diffusion_models/:
total 9765304
drwxr-xr-x  2 root root       4096 Aug 29 12:26 .
drwxr-xr-x 28 root root       4096 Aug 29 12:25 ..
-rw-r--r--  1 root root          0 Aug 29 12:25 put_diffusion_model_files_here
-rw-r--r--  1 root root 9999658848 Aug 29 12:27 wan2.2_ti2v_5B_fp16.safetensors

models/text_encoders/:
total 6578048
drwxr-xr-x  2 root root       4096 Aug 29 12:27 .
drwxr-xr-x 28 root root       4096 Aug 29 12:25 ..
-rw-r--r--  1 root root          0 Aug 29 12:25 put_text_encoder_files_here
-rw-r--r--  1 root root 6735906897 Aug 29 12:29 umt5_xxl_fp8_e4m3fn_scaled.safetensors

models/vae/:
total 1376384
drwxr-xr-x  2 root root       4096 Aug 29 12:27 .
drwxr-xr-x 28 root root       4096 Aug 29 12:25 ..
-rw-r--r--  1 root root      

> Downloads ~17GB total (diffusion model + VAE + text encoder). Takes a few minutes depending on Kaggle's bandwidth.

# Cell 3 — Get cloudflared binary

In [3]:
import os, subprocess
os.chdir("/kaggle/working/ComfyUI")
if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)
print("✅ Cloudflared listo en /kaggle/working/ComfyUI.")


-rwxr-xr-x 1 root root 39799316 Aug 14 12:24 cloudflared


# Cell 4 — Launch ComfyUI, verify it's alive

In [4]:
import subprocess, time, os, sys
os.chdir("/kaggle/working/ComfyUI")
print("🚀 Levantando servidor ComfyUI en GPU T4...")
subprocess.Popen([sys.executable, "main.py", "--listen", "0.0.0.0", "--port", "8188", "--highvram", "--fast"])
time.sleep(15)
subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "HTTP Status: %{http_code}\n", "http://localhost:8188"])


[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments


[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-08-29 12:29:12.159
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /root/ComfyUI
** ComfyUI Base Folder Path: /root/ComfyUI
** User directory: /root/ComfyUI/user
** ComfyUI-Manager config path: /root/ComfyUI/user/__manager/config.ini
** Log path: /root/ComfyUI/user/comfyui.log


[INFO] 
Prestartup times for custom nodes:
[INFO]    8.6 seconds: /root/ComfyUI/custom_nodes/ComfyUI-Manager
[INFO] 
[WARNING] WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations.
WARNING WARNING WARNING
If you are on nvidia 20 series and above it is required that you update your pytorch to cu130 or higher.

[INFO] Found comfy_kitchen backend hip: {'available': False, 'disabled': False, 'unavailable_reason': 'PyTorch ROCm/HIP runtime not available', 'capabilities': []}
[INFO] Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'convrot_w4a4_linear', 'dequantize_convrot_w4a4_weight', 'dequantize_int8_convrot_weight', 'dequantize_int8_convrot_weight_dtype', 'dequantize_int8_simple', 'dequantize_int8_simple_dtype', 'dequantize_

000


# Cell 5 — Start tunnel, get URL
> This will print a random `trycloudflare.com` URL — that's your ComfyUI address. Open it in a new tab, not this one.

In [5]:
import subprocess, time, re
subprocess.run(["pkill", "-f", "cloudflared"])
proc = subprocess.Popen(
    ["/kaggle/working/ComfyUI/cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
url = None
for _ in range(100):
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.3)
        continue
    m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break

print("\n" + "="*70)
print("🎉 ¡COMFYUI WAN 2.2 TI2V 5B ACTIVO!")
print(f"👉 ENLACE PÚBLICO: {url}")
print("="*70 + "\n")
print("🟢 Manteniendo el túnel y servidor activos... Dejar corriendo esta celda.")
try:
    while True:
        time.sleep(15)
except KeyboardInterrupt:
    print("🛑 Servidor detenido.")


2026-08-29T12:29:30Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-29T12:29:30Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-29T12:29:35Z INF +--------------------------------------------------------------------------------------------+
2026-08-29T12:29:35Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-29T12:29:35Z INF |  https://knowing-much-protocol-metals.trycloudflare.co